# Lahore Societies Map + Average Prices (Folium)

This notebook scans society maps in `../societies`, aggregates average listing prices from `../data` (Graana + Zameen), and renders an interactive Folium map saved to `../dha_price_heatmap.html`.

Steps:
- Discover shapefiles in `../societies/**/*.shp`
- Load scraped CSVs, parse prices to PKR, and compute per-society averages
- Join averages to society polygons and render a choropleth + popups


In [1]:
import os
import re
import math
import json
from pathlib import Path
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
from folium.features import GeoJsonTooltip

# Paths (notebook lives in notebooks/)
ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'data'
SOCIETIES_DIR = ROOT / 'societies'
OUTPUT_MAP = ROOT / 'dha_price_heatmap.html'

# Helper: clean price strings to numeric PKR
PRICE_PATTERNS = [
    (re.compile(r"([0-9]*\.?[0-9]+)\s*(crore|cr)\b", re.I), 1e7),
    (re.compile(r"([0-9]*\.?[0-9]+)\s*(million|m)\b", re.I), 1e6),
    (re.compile(r"([0-9]*\.?[0-9]+)\s*(lac|lakh|lac\.)\b", re.I), 1e5),
]

def parse_price_to_pkr(x):
    if pd.isna(x):
        return math.nan
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).strip().replace(',', '')
    # pure number
    try:
        return float(s)
    except ValueError:
        pass
    for pat, mult in PRICE_PATTERNS:
        m = pat.search(s)
        if m:
            return float(m.group(1)) * mult
    # plain PKR like 12345678 or with Rs
    m2 = re.search(r"([0-9]{3,})", s)
    if m2:
        return float(m2.group(1))
    return math.nan

# Helper: normalized society name from text
NORMALIZE_RE = re.compile(r"[^a-z0-9]+")

def normalize_name(s):
    if pd.isna(s):
        return None
    return NORMALIZE_RE.sub('-', str(s).lower()).strip('-')
